## Step 1: Load the Dataset and Select Required Columns

Read the source CSV, keep the two drug-name columns and interaction text, and inspect the dataset.


In [50]:
import pandas as pd

df = pd.read_csv("D:\\Generative AI Application Engineering bootcamp\\Final Project\\Data\\ddi_with_names.csv")

columns = ["Drug1_Name", "Drug2_Name", "Interaction"]
df = df[columns].copy()

print("Dataset shape:", df.shape)
print("\nMissing values:")
print(df.isna().sum())

df.head()

Dataset shape: (191808, 3)

Missing values:
Drug1_Name     185
Drug2_Name      82
Interaction      0
dtype: int64


,Drug1_Name,Drug2_Name,Interaction
0,Trioxsalen,Verteporfin,#Drug1 may increase the photosensitizing activ...
1,Aminolevulinic acid,Verteporfin,#Drug1 may increase the photosensitizing activ...
2,Titanium dioxide,Verteporfin,#Drug1 may increase the photosensitizing activ...
3,Tiaprofenic acid,Verteporfin,#Drug1 may increase the photosensitizing activ...
4,Cyamemazine,Verteporfin,#Drug1 may increase the photosensitizing activ...


## Step 2: Remove Records with Missing Values

Remove rows missing either drug name or the interaction text, then check the remaining records.


In [51]:
rows_before = len(df)

df = df.dropna(subset=columns).reset_index(drop=True)

print("Rows removed:", rows_before - len(df))
print("Dataset shape:", df.shape)
print("\nMissing values:")
print(df.isna().sum())

Rows removed: 267
Dataset shape: (191541, 3)

Missing values:
Drug1_Name     0
Drug2_Name     0
Interaction    0
dtype: int64


## Step 3: Replace Placeholders with Drug Names

Replace #Drug1 and #Drug2 in each interaction with the corresponding drug names.


In [52]:
df["Interaction"] = df.apply(
    lambda row: row["Interaction"]
    .replace("#Drug1", row["Drug1_Name"])
    .replace("#Drug2", row["Drug2_Name"]),
    axis=1
)

remaining_placeholders = df["Interaction"].str.contains(
    r"#Drug1|#Drug2", regex=True
).sum()

print("Remaining placeholders:", remaining_placeholders)
print("\nFirst interaction:")
print(df.loc[0, "Interaction"])

df.head()

Remaining placeholders: 0

First interaction:
Trioxsalen may increase the photosensitizing activities of Verteporfin.


,Drug1_Name,Drug2_Name,Interaction
0,Trioxsalen,Verteporfin,Trioxsalen may increase the photosensitizing a...
1,Aminolevulinic acid,Verteporfin,Aminolevulinic acid may increase the photosens...
2,Titanium dioxide,Verteporfin,Titanium dioxide may increase the photosensiti...
3,Tiaprofenic acid,Verteporfin,Tiaprofenic acid may increase the photosensiti...
4,Cyamemazine,Verteporfin,Cyamemazine may increase the photosensitizing ...


## Step 4: Trim Whitespace and Check Data Quality

Remove surrounding whitespace and count empty strings and exact duplicate rows.


In [53]:
for column in columns:
    df[column] = df[column].str.strip()

print("Empty strings:")
print(df[columns].eq("").sum())

print("\nExact duplicate rows:", df.duplicated(subset=columns).sum())

Empty strings:
Drug1_Name     0
Drug2_Name     0
Interaction    0
dtype: int64

Exact duplicate rows: 0


## Step 5: Inspect Interaction Directions and Multiple Records

Identify pairs with different interaction text in reverse directions, find ordered pairs with multiple interactions, and report the maximum distinct interaction count across both directions.


In [54]:
# Normalize values without changing df.
data = df.copy()
data["drug1"] = data["Drug1_Name"].str.strip().str.lower()
data["drug2"] = data["Drug2_Name"].str.strip().str.lower()
data["interaction"] = data["Interaction"].str.strip().str.lower()

# 1. Check records in reverse directions.
reverse_records = data.merge(
    data,
    left_on=["drug1", "drug2"],
    right_on=["drug2", "drug1"],
    suffixes=("_a", "_b")
)

reverse_records = reverse_records[
    (reverse_records["drug1_a"] < reverse_records["drug2_a"])
    & (reverse_records["interaction_a"] != reverse_records["interaction_b"])
]

reverse_pairs = reverse_records.drop_duplicates(
    subset=["drug1_a", "drug2_a"]
)

print("Drug pairs with different text in reverse directions:", len(reverse_pairs))

if not reverse_pairs.empty:
    example = reverse_pairs.iloc[0]

    print("\nReverse-direction example:")
    print(example["Drug1_Name_a"], "->", example["Drug2_Name_a"])
    print(example["Interaction_a"])
    print(example["Drug1_Name_b"], "->", example["Drug2_Name_b"])
    print(example["Interaction_b"])

# 2. Count ordered pairs with multiple distinct interactions.
counts = data.groupby(["drug1", "drug2"])["interaction"].nunique()
multiple_pairs = counts[counts > 1]

print("\nOrdered drug pairs with multiple interactions:", len(multiple_pairs))

if not multiple_pairs.empty:
    drug1, drug2 = multiple_pairs.index[0]

    example = data[
        (data["drug1"] == drug1)
        & (data["drug2"] == drug2)
    ].drop_duplicates(subset="interaction")

    print("\nMultiple-interaction example:")
    print(example.iloc[0]["Drug1_Name"], "->", example.iloc[0]["Drug2_Name"])

    for interaction in example["Interaction"]:
        print(interaction)

# 3. Print the maximum interaction count, combining both directions.
data["drug_a"] = data["drug1"].where(
    data["drug1"] <= data["drug2"], data["drug2"]
)
data["drug_b"] = data["drug2"].where(
    data["drug1"] <= data["drug2"], data["drug1"]
)

pair_counts = (
    data.groupby(["drug_a", "drug_b"])["interaction"]
    .nunique()
)
print(
    "\nMaximum distinct interactions for one drug pair (both directions):",
    int(pair_counts.max())
)

Drug pairs with different text in reverse directions: 117

Reverse-direction example:
Carbamazepine -> Doxorubicin
The metabolism of Doxorubicin can be increased when combined with Carbamazepine.
Doxorubicin -> Carbamazepine
The serum concentration of Carbamazepine can be increased when it is combined with Doxorubicin.

Ordered drug pairs with multiple interactions: 289

Multiple-interaction example:
Amodiaquine -> Celecoxib
The metabolism of Celecoxib can be decreased when combined with Amodiaquine.
The serum concentration of Celecoxib can be increased when it is combined with Amodiaquine.

Maximum distinct interactions for one drug pair (both directions): 2


## Step 6: Save the Cleaned Dataset and Count Unique Drugs

Export ddi_cleaned.csv and count unique drug names across both drug columns.


In [55]:
df.to_csv("ddi_cleaned.csv", index=False)

unique_drugs = (
    pd.concat([df["Drug1_Name"], df["Drug2_Name"]])
    .dropna()
    .str.strip()
    .str.lower()
    .nunique()
)

print("Saved file: ddi_cleaned.csv")
print("Total records:", len(df))
print("Unique drugs:", unique_drugs)

Saved file: ddi_cleaned.csv
Total records: 191541
Unique drugs: 1701


## Step 7: Install Retrieval Dependencies

Install the packages needed for document preparation, embeddings, and Chroma storage.


In [9]:
%pip install -q langchain-core langchain-huggingface langchain-chroma chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 115.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/

## Step 8: Create Documents with Interaction Metadata

Use both drug names as document content and store the interaction, drug names, source, and record ID in metadata.


In [10]:
import numpy as np
import torch

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma


documents = [
    Document(
        page_content=f"{row.Drug1_Name} {row.Drug2_Name}",
        metadata={
            "record_id": index + 1,
            "drug1_name": row.Drug1_Name,
            "drug2_name": row.Drug2_Name,
            "interaction": row.Interaction,
            "source": "ddi_cleaned.csv"
        }
    )
    for index, row in enumerate(df.itertuples(index=False))
]

print(f"Total documents: {len(documents):,}")
print("\nFirst document:")
print(documents[0].page_content)
print("\nMetadata:")
print(documents[0].metadata)

Total documents: 191,541

First document:
Trioxsalen Verteporfin

Metadata:
{'record_id': 1, 'drug1_name': 'Trioxsalen', 'drug2_name': 'Verteporfin', 'interaction': 'Trioxsalen may increase the photosensitizing activities of Verteporfin.', 'source': 'ddi_cleaned.csv'}


## Step 9: Load the Embedding Model

Initialize BAAI/bge-small-en-v1.5 with normalized embeddings and select CUDA when available.


In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": device},
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 128
    },
    show_progress=True
)

print(f"Device: {device}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Device: cuda


## Step 10: Build and Persist the Chroma Database

Embed the documents and save them in the ddi_interactions collection under ddi_chroma_db.


In [12]:
vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="ddi_interactions",
    persist_directory="./ddi_chroma_db"
)

print("Vector database created successfully.")

Batches:   0%|          | 0/43 [00:00<?, ?it/s]

Batches:   0%|          | 0/43 [00:00<?, ?it/s]

Batches:   0%|          | 0/43 [00:00<?, ?it/s]

Batches:   0%|          | 0/43 [00:00<?, ?it/s]

Batches:   0%|          | 0/43 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/43 [00:00<?, ?it/s]

Batches:   0%|          | 0/43 [00:00<?, ?it/s]

Batches:   0%|          | 0/43 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Vector database created successfully.


## Step 11: Verify the Stored Document Count

Check how many documents are stored in the Chroma collection.


In [13]:
print("Stored documents:", vector_store._collection.count())

Stored documents: 191541


## Step 12: Back Up the Database in Colab (Optional)

The commented code creates and downloads a ZIP containing the database folder when running in Colab.


In [ ]:
# import shutil
# from google.colab import files

# archive_path = shutil.make_archive(
#     "/content/ddi_chroma_db_backup",
#     "zip",
#     root_dir="/content",
#     base_dir="ddi_chroma_db"
# )

# files.download(archive_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Step 13: Reload the Saved Database for Testing

Load the extracted database with the same embedding model instead of rebuilding the stored vectors for each test.


In [2]:
from pathlib import Path
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

DB_PATH = Path(r"ddi_chroma_db_backup\ddi_chroma_db")

if not (DB_PATH / "chroma.sqlite3").is_file():
    raise FileNotFoundError(
        "Database not found. Extract the ZIP and check DB_PATH."
    )

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cuda"},
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 128
    },
    show_progress=False
)

vector_store = Chroma(
    collection_name="ddi_interactions",
    persist_directory=str(DB_PATH),
    embedding_function=embeddings,
    create_collection_if_not_exists=False
)

print("Vector database loaded successfully.")
print("Stored documents:", vector_store._collection.count())

C:\Users\misha\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3502.63it/s]


Vector database loaded successfully.
Stored documents: 191541


## Step 14: Test Retrieval for a Sample Drug Pair

Create a question from the first dataset row and display the top three retrieved records.


In [7]:
question = (
    f"What is the interaction between "
    f"{df.iloc[0]['Drug1_Name']} and {df.iloc[0]['Drug2_Name']}?"
)

results = vector_store.similarity_search(question, k=3)

print("Question:", question)

for position, doc in enumerate(results, start=1):
    print(f"\nResult {position}")
    print(doc.page_content)
    print("Interaction:", doc.metadata["interaction"])
    print("Metadata:", doc.metadata)

Question: What is the interaction between Trioxsalen and Verteporfin?

Result 1
Trioxsalen Verteporfin
Interaction: Trioxsalen may increase the photosensitizing activities of Verteporfin.
Metadata: {'drug1_name': 'Trioxsalen', 'drug2_name': 'Verteporfin', 'record_id': 1, 'interaction': 'Trioxsalen may increase the photosensitizing activities of Verteporfin.', 'source': 'ddi_cleaned.csv'}

Result 2
Methoxsalen Verteporfin
Interaction: Methoxsalen may increase the photosensitizing activities of Verteporfin.
Metadata: {'interaction': 'Methoxsalen may increase the photosensitizing activities of Verteporfin.', 'drug1_name': 'Methoxsalen', 'record_id': 7, 'drug2_name': 'Verteporfin', 'source': 'ddi_cleaned.csv'}

Result 3
Docetaxel Verteporfin
Interaction: The risk or severity of adverse effects can be increased when Docetaxel is combined with Verteporfin.
Metadata: {'source': 'ddi_cleaned.csv', 'record_id': 113845, 'drug1_name': 'Docetaxel', 'interaction': 'The risk or severity of adverse e

## Step 15: Find Pairs Recorded in Both Directions

Find reversed drug pairs and inspect records whose interaction text differs between directions.


In [23]:
pairs = df[["Drug1_Name", "Drug2_Name", "Interaction"]].copy()

pairs["drug1_key"] = pairs["Drug1_Name"].str.strip().str.lower()
pairs["drug2_key"] = pairs["Drug2_Name"].str.strip().str.lower()

reversed_pairs = pairs.merge(
    pairs,
    left_on=["drug1_key", "drug2_key"],
    right_on=["drug2_key", "drug1_key"],
    suffixes=("_forward", "_reverse")
)

# Exclude self-pairs and display each pair once.
reversed_pairs = reversed_pairs[
    reversed_pairs["drug1_key_forward"]
    < reversed_pairs["drug2_key_forward"]
].copy()

different_interactions = reversed_pairs[
    reversed_pairs["Interaction_forward"].str.strip().str.lower()
    != reversed_pairs["Interaction_reverse"].str.strip().str.lower()
]

print("Pairs of records found in both directions:", len(reversed_pairs))
print("Pairs of records with different interaction text:", len(different_interactions))

for row in different_interactions.head(2).itertuples(index=False):
    print(f"\n{row.Drug1_Name_forward} -> {row.Drug2_Name_forward}")
    print(row.Interaction_forward)

    print(f"\n{row.Drug1_Name_reverse} -> {row.Drug2_Name_reverse}")
    print(row.Interaction_reverse)
    print("-" * 80)

Pairs of records found in both directions: 117
Pairs of records with different interaction text: 117

Carbamazepine -> Doxorubicin
The metabolism of Doxorubicin can be increased when combined with Carbamazepine.

Doxorubicin -> Carbamazepine
The serum concentration of Carbamazepine can be increased when it is combined with Doxorubicin.
--------------------------------------------------------------------------------

Cyclophosphamide -> Vincristine
Cyclophosphamide may increase the cardiotoxic activities of Vincristine.

Vincristine -> Cyclophosphamide
The serum concentration of Cyclophosphamide can be increased when it is combined with Vincristine.
--------------------------------------------------------------------------------


## Step 16: Test Retrieval for a Pair with Different Directional Records

Ask about one of the identified pairs and inspect whether its directional records appear in the top three results.


In [25]:
row = different_interactions.iloc[0]

question = (
    f"What is the interaction between "
    f"{row['Drug1_Name_forward']} and {row['Drug2_Name_forward']}?"
)

results = vector_store.similarity_search(question, k=3)

print("Question:", question)

for position, doc in enumerate(results, start=1):
    print(f"\nResult {position}")
    print(doc.page_content)
    print("Interaction:", doc.metadata["interaction"])
    print("Metadata:", doc.metadata)

Question: What is the interaction between Carbamazepine and Doxorubicin?

Result 1
Carbamazepine Doxorubicin
Interaction: The metabolism of Doxorubicin can be increased when combined with Carbamazepine.
Metadata: {'drug1_name': 'Carbamazepine', 'interaction': 'The metabolism of Doxorubicin can be increased when combined with Carbamazepine.', 'record_id': 2860, 'drug2_name': 'Doxorubicin', 'source': 'ddi_cleaned.csv'}

Result 2
Doxorubicin Carbamazepine
Interaction: The serum concentration of Carbamazepine can be increased when it is combined with Doxorubicin.
Metadata: {'source': 'ddi_cleaned.csv', 'interaction': 'The serum concentration of Carbamazepine can be increased when it is combined with Doxorubicin.', 'record_id': 159858, 'drug2_name': 'Carbamazepine', 'drug1_name': 'Doxorubicin'}

Result 3
Clotiazepam Doxorubicin
Interaction: The metabolism of Doxorubicin can be decreased when combined with Clotiazepam.
Metadata: {'source': 'ddi_cleaned.csv', 'interaction': 'The metabolism of

## Step 17: Test Retrieval for a Pair with Multiple Interactions

Select an ordered pair with multiple distinct interactions, display its recorded interactions, and inspect the top three retrieved results.


In [13]:
pairs = df.copy()

pairs["drug1_key"] = pairs["Drug1_Name"].str.strip().str.lower()
pairs["drug2_key"] = pairs["Drug2_Name"].str.strip().str.lower()
pairs["interaction_key"] = pairs["Interaction"].str.strip().str.lower()

interaction_counts = (
    pairs.groupby(["drug1_key", "drug2_key"])["interaction_key"]
    .nunique()
)

multiple_interactions = interaction_counts[
    interaction_counts > 1
]

print(
    "Ordered drug pairs with multiple interactions:",
    len(multiple_interactions)
)

if multiple_interactions.empty:
    print("No matching cases were found.")
else:
    drug1_key, drug2_key = multiple_interactions.index[0]

    expected_records = pairs[
        (pairs["drug1_key"] == drug1_key)
        & (pairs["drug2_key"] == drug2_key)
    ].drop_duplicates(subset="interaction_key")

    print("\nRecorded interactions for this ordered pair:")
    for row in expected_records.itertuples(index=False):
        print(row.Interaction)

    first_record = expected_records.iloc[0]

    question = (
        f"What is the interaction between "
        f"{first_record['Drug1_Name']} and "
        f"{first_record['Drug2_Name']}?"
    )

    results = vector_store.similarity_search(question, k=3)

    print("\nQuestion:", question)

    for position, doc in enumerate(results, start=1):
        print(f"\nResult {position}")
        print(doc.page_content)
        print("Interaction:", doc.metadata["interaction"])
        print("Metadata:", doc.metadata)

Ordered drug pairs with multiple interactions: 289

Recorded interactions for this ordered pair:
The metabolism of Celecoxib can be decreased when combined with Amodiaquine.
The serum concentration of Celecoxib can be increased when it is combined with Amodiaquine.

Question: What is the interaction between Amodiaquine and Celecoxib?

Result 1
Amodiaquine Celecoxib
Interaction: The metabolism of Celecoxib can be decreased when combined with Amodiaquine.
Metadata: {'drug2_name': 'Celecoxib', 'drug1_name': 'Amodiaquine', 'record_id': 67417, 'source': 'ddi_cleaned.csv', 'interaction': 'The metabolism of Celecoxib can be decreased when combined with Amodiaquine.'}

Result 2
Amodiaquine Celecoxib
Interaction: The serum concentration of Celecoxib can be increased when it is combined with Amodiaquine.
Metadata: {'record_id': 177582, 'drug2_name': 'Celecoxib', 'source': 'ddi_cleaned.csv', 'drug1_name': 'Amodiaquine', 'interaction': 'The serum concentration of Celecoxib can be increased when it

## Step 18: Prepare English and Arabic Evaluation Questions

Sample 1,000 rows and generate five questions per row in each language using the same sample and the recorded interaction as the expected result.


In [45]:
eval_rows = df.sample(n=1000, random_state=42)

english_templates = [
    "Can I take {drug1} and {drug2} together?",
    "I'm taking {drug1}. Does it interact with {drug2}?",
    "What could happen if I use {drug1} with {drug2}?",
    "Is there an interaction between {drug2} and {drug1}?",
    "Do {drug2} and {drug1} affect each other?"
]

arabic_templates = [
    "أقدر آخذ {drug1} و {drug2} مع بعض؟",
    "أنا أستخدم {drug1} ، هل يتفاعل مع {drug2}؟",
    "وش ممكن يصير لو استخدمت {drug1} مع {drug2}؟",
    "هل فيه تفاعل بين {drug2} و {drug1}؟",
    "هل {drug2} و {drug1} يأثرون على بعض؟"
] 


def prepare_questions(rows, templates):
    return [
        {
            "case_id": case_id,
            "template_id": template_id,
            "question": template.format(
                drug1=row.Drug1_Name,
                drug2=row.Drug2_Name
            ),
            "drug1_name": row.Drug1_Name.strip().lower(),
            "drug2_name": row.Drug2_Name.strip().lower(),
            "expected_interaction": row.Interaction.strip().lower()
        }
        for case_id, row in enumerate(
            rows.itertuples(index=False), start=1
        )
        for template_id, template in enumerate(templates, start=1)
    ]


english_questions = prepare_questions(eval_rows, english_templates)
arabic_questions = prepare_questions(eval_rows, arabic_templates)

print("Evaluation rows:", len(eval_rows))
print("English questions:", len(english_questions))
print("Arabic questions:", len(arabic_questions))

print("\nEnglish examples:")
for item in english_questions[:5]:
    print(f"{item['template_id']}: {item['question']}")

print("\nArabic examples:")
for item in arabic_questions[:5]:
    print(f"{item['template_id']}: {item['question']}")


Evaluation rows: 1000
English questions: 5000
Arabic questions: 5000

English examples:
1: Can I take Cortisone acetate and Minaprine together?
2: I'm taking Cortisone acetate. Does it interact with Minaprine?
3: What could happen if I use Cortisone acetate with Minaprine?
4: Is there an interaction between Minaprine and Cortisone acetate?
5: Do Minaprine and Cortisone acetate affect each other?

Arabic examples:
1: أقدر آخذ Cortisone acetate و Minaprine مع بعض؟
2: أنا أستخدم Cortisone acetate ، هل يتفاعل مع Minaprine؟
3: وش ممكن يصير لو استخدمت Cortisone acetate مع Minaprine؟
4: هل فيه تفاعل بين Minaprine و Cortisone acetate؟
5: هل Minaprine و Cortisone acetate يأثرون على بعض؟


## Step 19: Define Retrieval Evaluation Metrics

Compare retrieved interaction text with the expected text after trimming whitespace and lowercasing. Calculate Hit@1, Hit@3, and MRR@3 overall and by template.

In [47]:
def evaluate_retrieval(questions):
    evaluation_results = []
    previous_progress = embeddings.show_progress
    embeddings.show_progress = False

    try:
        for item in questions:
            retrieved = vector_store.similarity_search(
                item["question"],
                k=3
            )

            rank = None

            for position, doc in enumerate(retrieved, start=1):
                interaction = doc.metadata["interaction"].strip().lower()

                if (
                interaction == item["expected_interaction"]
                ):
                    rank = position
                    break

            evaluation_results.append({
                **item,
                "rank": rank,
                "Hit@1": int(rank == 1),
                "Hit@3": int(rank is not None),
                "MRR@3": 1 / rank if rank else 0
            })
    finally:
        embeddings.show_progress = previous_progress

    evaluation_df = pd.DataFrame(evaluation_results)
    metrics = ["Hit@1", "Hit@3", "MRR@3"]

    print("Overall metrics:")
    print(evaluation_df[metrics].mean().round(3))

    print("\nMetrics by template:")
    print(
        evaluation_df.groupby("template_id")[metrics]
        .mean()
        .round(3)
    )

    return evaluation_df

## Step 20: Run the English Retrieval Evaluation

Evaluate the 5,000 English questions and keep the per-question results.


In [48]:
english_evaluation = evaluate_retrieval(english_questions)

Overall metrics:
Hit@1    0.910
Hit@3    0.913
MRR@3    0.912
dtype: float64

Metrics by template:
             Hit@1  Hit@3  MRR@3
template_id                     
1            0.910  0.912  0.911
2            0.903  0.908  0.905
3            0.912  0.915  0.914
4            0.912  0.914  0.913
5            0.914  0.916  0.915


## Step 21: Run the Arabic Retrieval Evaluation

Evaluate the 5,000 Arabic questions separately using the same sampled rows. Drug names remain in their dataset language.


In [49]:
arabic_evaluation = evaluate_retrieval(arabic_questions)

Overall metrics:
Hit@1    0.906
Hit@3    0.909
MRR@3    0.907
dtype: float64

Metrics by template:
             Hit@1  Hit@3  MRR@3
template_id                     
1            0.890  0.893  0.891
2            0.906  0.907  0.906
3            0.912  0.914  0.913
4            0.918  0.918  0.918
5            0.904  0.911  0.907
